In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import torch
import pickle

from sentence_transformers import SentenceTransformer,InputExample,losses,util

from torch.utils.data import DataLoader

from src.metric import *

In [ ]:
with open('../data/cleaned/clean_train_df.pkl','rb') as f:
    train_df=pickle.load(f)
    
with open('../data/cleaned/clean_val_df.pkl','rb') as f:
    val_df=pickle.load(f)
        
with open('../data/cleaned/clean_test_df.pkl','rb') as f:
    test_df=pickle.load(f)
    


In [ ]:
device="cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
model_path='../models/bi_encoder'
os.makedirs(model_path,exist_ok=True)

In [ ]:
bi_encoder= SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2',device=device)

In [ ]:
label_to_score = {0: 0.0, 1: 0.5, 2: 1.0}

In [ ]:
bi_train_examples=[
    InputExample(texts=[j,r],label=float(label_to_score[l]))
    for r,j,l in zip(train_df['resume_text'],train_df['job_description_text'],train_df['label'])
]

bi_train_dataloader=DataLoader(bi_train_examples,shuffle=True,batch_size=64)

bi_train_loss=losses.CoSENTLoss(bi_encoder)

In [ ]:
epochs = 4
best_score = float('-inf')
min_delta=0.01

print("\tTraining Phase")
for epoch in range(1, epochs + 1):
    print(f"Epoch: {epoch}----------")

    bi_encoder.fit(
        train_objectives=[(bi_train_dataloader, bi_train_loss)],
        epochs=1,
        warmup_steps=int(len(bi_train_dataloader) * epochs * 0.1),
        show_progress_bar=True
    )

    val_resume_emb=bi_encoder.encode(
        val_df['resume_text'].tolist(),
        batch_size=64,convert_to_tensor=True,show_progress_bar=False
    )
    val_jd_emb =bi_encoder.encode(
        val_df['job_description_text'].tolist(),
        batch_size=64,convert_to_tensor=True,show_progress_bar=False
    )

    scores = torch.cosine_similarity(val_resume_emb, val_jd_emb).cpu().numpy()

    metrics = model_evaluation(scores, val_df, 'job_description_text')

    print("NDCG:",metrics['ndcg_val'])
    print("MAP:",metrics['map_score'])

    final_score = (0.6*metrics['ndcg_val'] +
                   0.3*metrics['map_score'] +
                   0.1*(metrics['mrr_score'] + metrics['topk_score']))

    if final_score > best_score+min_delta:
        best_score=final_score
        bi_encoder.save("../models/baseline/bi_encoder
                        ")
        
        

print("\tInference Phase")
val_resume_emb=bi_encoder.encode(val_df['resume_text'].tolist(),
                              batch_size=64,convert_to_tensor=True,
                              show_progress_bar=True)
val_jd_emb = bi_encoder.encode(val_df['job_description_text'].tolist(),
                              batch_size=64,convert_to_tensor=True,
                              show_progress_bar=True)

scores = torch.cosine_similarity(val_resume_emb,val_jd_emb).cpu().numpy()


In [ ]:
metrics=model_evaluation(scores,val_df,'job_description_text')
print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])

In [ ]:
eval_df=val_df.copy()
eval_df['score']=scores
print("\nScore spread within groups:")
spreads = []
for jd, group in eval_df.groupby('job_description_text'):
    if len(group) > 1:
        spreads.append(group['score'].max() - group['score'].min())

print(f"Mean spread: {np.mean(spreads):.3f}")
print(f"% groups with spread < 0.1:"f"{(np.array(spreads) < 0.1).mean():.3f}")

In [ ]:
test_resume_emb=bi_encoder.encode(test_df['resume_text'].tolist(),
                              batch_size=64,convert_to_tensor=True,
                              show_progress_bar=True)
test_jd_emb = bi_encoder.encode(test_df['job_description_text'].tolist(),
                              batch_size=64,convert_to_tensor=True,
                              show_progress_bar=True)

scores = torch.cosine_similarity(test_resume_emb,test_jd_emb).cpu().numpy()

metrics=model_evaluation(scores,test_df,'job_description_text')

In [ ]:
print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])